# Regression - Cross-Validation

This notebook introduces the Scikit-Learn interface for cross-validation.

## Setup

Load the packages and configure environment.

In [ ]:
import numpy as np
import pandas as pd

Using the Boston data set, which contains housing and neighborhood data for 506 census tracts in the Boston area. We'll predict `crim` (per capita crime rate) from neighborhood characteristics like property tax rates, student-teacher ratios, and proximity to employment centers.

In [ ]:
# download the data set directly from the web using pandas
url = "https://raw.githubusercontent.com/olearydj/INSY7120/refs/heads/main/notebooks/data/Boston.csv"
boston = pd.read_csv(url)
# get the predictors of interest
X = boston.loc[:,'zn':]
y = boston['crim']

In [ ]:
X.head()

In [ ]:
y.head()

## The Problem with Training Error

In previous notebooks, we've evaluated models on the same data used to train them. As discussed in our bias-variance lecture, this **training error** is optimistically biased—it underestimates how well the model will perform on new data.

To get an honest estimate of model performance, we need to evaluate on data the model hasn't seen during training.

## Train-Test Split

The simplest approach: hold out a portion of data for testing. Let's first do this manually with pandas to understand what's happening.

In [ ]:
from sklearn.linear_model import LinearRegression

# Manual split: shuffle the data, then slice
np.random.seed(42)
shuffled_idx = np.random.permutation(len(X))

# 70% train, 30% test
split_point = int(len(X) * 0.7)
train_idx = shuffled_idx[:split_point]
test_idx = shuffled_idx[split_point:]

# Use iloc to select rows by position
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print(f"Training set: {len(X_train)} observations")
print(f"Test set: {len(X_test)} observations")
X_train.head()

In [ ]:
# Fit on training data only
model = LinearRegression()
model.fit(X_train, y_train)

# Compare training vs test performance
print(f"Training R²: {model.score(X_train, y_train):.4f}")
print(f"Test R²: {model.score(X_test, y_test):.4f}")

Note on `model.score()`: This is a convenience method that returns R² for regression models. It's equivalent to calling `r2_score(y, model.predict(X))` but more concise. In earlier notebooks we used the explicit `predict()` then `r2_score()` approach—both give identical results.

Here the training R² is *lower* than test R², the opposite of what we typically expect. This isn't a sign of underfitting; it's simply that this particular random split happened to put "easier" observations in the test set. This illustrates why a single split can be misleading—the estimate depends heavily on which observations end up where.

### Using `train_test_split`

The manual approach works but is verbose and error-prone. Sklearn's `train_test_split` handles shuffling, splitting, and index alignment in one call.

In [ ]:
from sklearn.model_selection import train_test_split

# Same split, much cleaner
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

print(f"Training R²: {model.score(X_train, y_train):.4f}")
print(f"Test R²: {model.score(X_test, y_test):.4f}")

The test R² gives us a more realistic estimate of how the model will perform on new data than training R² alone.

Note that training is again less than test; seeing this twice should give us pause. The `crim` variable is highly skewed with extreme outliers. Since the training set is larger, it's more likely to contain these hard-to-predict outliers, dragging down training R². This is another reason single splits can mislead - data characteristics can systematically bias the comparison.

In [ ]:
# Verify: crim is highly skewed with extreme outliers
from scipy.stats import skew
print(f"Skewness of crim: {skew(y):.2f}")  # >1 indicates strong right skew
print(f"Median: {y.median():.2f}, Mean: {y.mean():.2f}, Max: {y.max():.2f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(y, bins=30, edgecolor='black')
axes[0].axvline(y.median(), color='blue', linestyle='--', label=f"Median: {y.median():.1f}")
axes[0].axvline(y.mean(), color='red', linestyle='--', label=f"Mean: {y.mean():.1f}")
axes[0].set_xlabel('Crime Rate')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Crime Rate')
axes[0].legend()

# Box plot
axes[1].boxplot(y, vert=True)
axes[1].set_ylabel('Crime Rate')
axes[1].set_title('Box Plot Showing Outliers')

plt.tight_layout()
plt.show()

Further evidence of the value of exploratory data analysis prior to modeling!

## Limitation of a Single Split

A single train-test split has drawbacks:
- The test estimate depends heavily on *which* observations end up in the test set
- We "waste" 30% of our data—it's never used for training
- If we want to compare multiple models or tune hyperparameters, we'd need to use the test set repeatedly, which defeats its purpose

## Leave-One-Out Cross-Validation (LOOCV)

One extreme solution: use *every* observation as a test set exactly once. With n observations, fit n models—each trained on n-1 observations and tested on the 1 held out.

In [ ]:
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.linear_model import LinearRegression

loo = LeaveOneOut()
model = LinearRegression()

# This fits 506 models (one per observation)!
# We use neg_mean_squared_error instead of r2 because R² is undefined for a
# single observation: SS_tot = sum((y - y_mean)²) = 0 when there's only one y.
scores_loo = cross_val_score(model, X, y, cv=loo, scoring='neg_mean_squared_error')
mean_mse = -scores_loo.mean()
print(f"LOOCV: {len(scores_loo)} folds")
print(f"Mean MSE: {mean_mse:.4f}")
print(f"RMSE:     {np.sqrt(mean_mse):.4f}")

LOOCV has **low bias** (each model trains on nearly all the data) but **high variance**—the n training sets share n-2 observations, so they're highly correlated. Averaging correlated estimates doesn't reduce variance as effectively as averaging independent ones.

It's also **computationally expensive**: fitting n models can be slow for large datasets. For these reasons, k-fold CV (typically k=5 or k=10) is usually preferred in practice.

## K-Fold Cross-Validation

**Cross-validation** addresses these issues by systematically rotating which data is used for training vs. validation:

1. Split the data into *k* equal-sized "folds"
2. For each fold: train on *k-1* folds, evaluate on the held-out fold
3. Average the *k* scores for a more stable estimate

With 5-fold CV, every observation gets used for validation exactly once, and we get 5 performance estimates instead of 1. This reduces dependence on any particular split.

### Why Do We Need This?

Consider a concrete example: in the previous lecture, we saw that `PolynomialFeatures` can generate degree-1, degree-2, or degree-3 terms. How do we decide which degree is best?

- Training error won't help: higher degree always fits training data better (or at least as well)
- A single test split is risky - our choice might depend on which observations happened to land in the test set

With cross-validation, we can fit each polynomial degree *k* times, get *k* test scores for each, and compare the averages. The degree with the best average CV score is our choice, and we have more confidence because it's based on multiple splits, not just one lucky (or unlucky) draw.

### Data Setup

In practice, we still hold out a final test set that's *never* used during model development - it gives us one final, unbiased estimate after we've made all our modeling decisions. Cross-validation happens on the remaining data.

In [ ]:
from sklearn.model_selection import train_test_split

# We call this X_train_val (not X_train) to emphasize that this data serves
# two purposes: training within each CV fold, and validation across folds.
# X_test remains completely untouched until final evaluation.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

## `cross_val_score`

Use for quick evaluations of test error with a single metric.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

model = LinearRegression()

# Simple cross-validation with default 5-fold
scores = cross_val_score(model, X_train_val, y_train_val, cv=5, scoring='r2')
print(scores)
print(f"\nMean R²: {scores.mean():.4f} (+/- {scores.std():.4f})")

Each value is the R² from one fold—the model was trained on 4 folds and tested on the 5th. The variation across folds (here ranging from ~0.36 to ~0.50) shows how sensitive our performance estimate is to which data is held out. The mean provides a more stable estimate than any single split, and the standard deviation quantifies uncertainty.

## `cross_validate`

For more thorough analysis. Returns a dict of scores and timing information for both test and train.

In [ ]:
from sklearn.model_selection import cross_validate
# use pretty print to make structures easier to read
from pprint import pprint as pp

# Multiple metrics and training scores
results = cross_validate(model, X_train_val, y_train_val, cv=5, 
                        scoring=['r2', 'neg_root_mean_squared_error', 'neg_median_absolute_error'],
                        return_train_score=True)
pp(results)

Note that the scores are the same. By default, these methods create folds based on the order of the data provided. For example, if there are 5 folds, the first 20% of the data is assigned to the first fold, the second 20% to the second, and so on. They inherit the random order of the `train_test_split` (and its seed). In SKL syntax, they default to `shuffle=False`).

For greater control over this process, you can create a CV splitter with `shuffle=True` and fixed random state.

In [ ]:
from sklearn.model_selection import KFold

# Create a CV splitter with explicit shuffle control
kf = KFold(n_splits=5, shuffle=True, random_state=42)

**Why `shuffle=True`?** Without shuffling, folds are purely positional (fold 1 = first 20% of rows, etc.). If the data has any ordering—sorted by region, date, customer ID—the folds won't be representative samples. Shuffling randomizes row order so each fold is a random sample of the whole dataset.

**Why `random_state`?** Makes the shuffle reproducible. Same seed = same folds every time, which matters for debugging and fair model comparison (all models evaluated on identical folds).

This code does not perform the split, it only initializes a splitter, which can then be used in `cross_val_score` or `cross_validate`.

In [ ]:
# Pass this CV splitter to cross_val_score
# scoring='r2' is also the default for regression models
scores = cross_val_score(model, X_train_val, y_train_val, cv=kf, scoring='r2')
print(scores)

## Fit and Evaluate the Final Model on Test Data

In a real workflow, you'd use CV to compare multiple model configurations—say, polynomial degrees 1, 2, and 3—and pick the one with the best average CV score. Here we only have one model (plain LinearRegression), so there's nothing to "select." We're just demonstrating the final step of the process.

Once you've chosen your model configuration, you:
1. **Refit** on all the training data (CV only used subsets for each fold)
2. **Evaluate once** on the held-out test set for a final, unbiased performance estimate

In [ ]:
# Refit on ALL training data (not just k-1 folds)
final_model = LinearRegression()
final_model.fit(X_train_val, y_train_val)

# Final evaluation on the held-out test set
test_score = final_model.score(X_test, y_test)
print(f"Final model R² on test set: {test_score:.4f}")

# You can also calculate other metrics on the test set
from sklearn.metrics import mean_squared_error, median_absolute_error

y_pred = final_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = median_absolute_error(y_test, y_pred)

print(f"Root Mean Squared Error on test set: {rmse:.4f}")
print(f"Median Absolute Error on test set: {mae:.4f}")

# Optional: Examine the model coefficients
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': final_model.coef_
}).sort_values('Coefficient', ascending=False)

print("\nModel Coefficients:")
print(coefficients)

## End-to-End Example: Model Selection with CV

Let's put it all together. We'll compare polynomial regression models of degree 1, 2, and 3, select the best using CV, then evaluate on the held-out test set.

To keep feature counts manageable, we'll use a single predictor (`lstat` - % lower status population). With 13 predictors and degree 3, we'd generate ~560 features from only ~350 observations—a recipe for overfitting.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Use single predictor to keep polynomial features manageable
X_single = X_train_val[['lstat']]
X_single_test = X_test[['lstat']]

# Compare polynomial degrees using CV
degrees = [1, 2, 3]
cv_results = {}

for d in degrees:
    # Pipeline: polynomial features → linear regression
    pipe = make_pipeline(PolynomialFeatures(d, include_bias=False), LinearRegression())
    scores = cross_val_score(pipe, X_single, y_train_val, cv=5, scoring='r2')
    cv_results[d] = scores.mean()
    print(f"Degree {d}: Mean CV R² = {scores.mean():.4f} (+/- {scores.std():.4f})")

# Select best
best_degree = max(cv_results, key=cv_results.get)
print(f"\nBest: degree {best_degree}")

In [ ]:
# Refit best model on ALL training data
final_pipe = make_pipeline(PolynomialFeatures(best_degree, include_bias=False), LinearRegression())
final_pipe.fit(X_single, y_train_val)

# Final evaluation on held-out test set
test_r2 = final_pipe.score(X_single_test, y_test)
print(f"Final model (degree {best_degree}) R² on test set: {test_r2:.4f}")

This is the complete workflow: use CV to compare candidates, select the best, refit on all training data, evaluate once on test.

## Repeated K-Fold Cross-Validation

A single run of k-fold CV gives us k scores, but how stable is that estimate? If we ran k-fold again with a different random shuffle, would we get similar results?

**Repeated k-fold CV** runs the entire k-fold process multiple times, each with a different random partition of the data. This gives us:
- A distribution of scores (not just k, but k × n_repeats)
- A "grand mean" that averages across all runs
- A standard deviation that reflects uncertainty in our estimate

In [ ]:
from sklearn.model_selection import KFold, RepeatedKFold

# Single 5-fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores_single = cross_val_score(model, X_train_val, y_train_val, cv=kf, scoring='r2')

print("Single 5-Fold CV:")
print(f"  Scores: {scores_single}")
print(f"  Mean: {scores_single.mean():.4f}, Std: {scores_single.std():.4f}")

In [ ]:
# Repeated 5-fold CV (10 repetitions = 50 total evaluations)
rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)
scores_repeated = cross_val_score(model, X_train_val, y_train_val, cv=rkf, scoring='r2')

print("Repeated 5-Fold CV (10 repeats):")
print(f"  Number of scores: {len(scores_repeated)}")
print(f"  Grand Mean: {scores_repeated.mean():.4f}, Std: {scores_repeated.std():.4f}")

The repeated CV gives us 50 scores instead of 5. The grand mean is our best estimate of model performance, and the standard deviation tells us how much that estimate varies depending on how the data is partitioned.

This is especially valuable when:
- You have limited data and want to squeeze more information from it
- You're comparing models and need confidence that differences are real, not due to a lucky/unlucky split
- You want to report uncertainty in your performance estimates